# RLDX-1: PushT Training/Eval + LIBERO Benchmark

End-to-end walkthrough for the `Rldx1` policy in Physical AI Studio:

| # | Section | What you'll do |
|---|---------|----------------|
| 1 | **Train on PushT** | Fine-tune `RLWRLD/RLDX-1-PT` on `lerobot/pusht`, track the best checkpoint by validation success rate |
| 2 | **Eval on PushT** | Run `PushTBenchmark` on the best checkpoint, record and display rollout videos |
| 3 | **LIBERO benchmark** | Evaluate `RLDX-1-FT-LIBERO` on a LIBERO task, record and display videos |

For the RoboCasa Kitchen benchmark, see the separate
[`benchmark/robocasa.ipynb`](../benchmark/robocasa.ipynb) notebook -- RoboCasa
needs its own Python environment (dependency conflicts with this repo's
`lerobot`/`[libero]` pins), so it runs under its own dedicated kernel instead
of this one.

### Prerequisites

- `physicalai-train` installed (`uv sync` from `library/`), CUDA GPU recommended.
- All sections in this notebook run in **this** notebook kernel.

---
## 1. Train RLDX-1 on PushT

`lerobot/pusht`: fps 10, 206 episodes, single top-down camera, 2-D state/action
(absolute agent position, no gripper). We fine-tune from the `RLWRLD/RLDX-1-PT`
pretrained checkpoint with `NON_EEF`/`ABSOLUTE` 2-D actions.

The config below is a **quick smoke-test run** (`max_epochs=5`,
`limit_train_batches=5`) so the whole notebook finishes quickly end-to-end.
For a real training run, increase `max_epochs`, drop `limit_train_batches`/
`limit_val_batches`, and consider a larger `train_batch_size`.

In [2]:
# Copyright (C) 2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

import subprocess
from pathlib import Path

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
REPO_ROOT

PosixPath('/home/yuchunli/git/physical-ai-studio')

In [ ]:
import multiprocessing

from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint

from physicalai.data import LeRobotDataModule
from physicalai.gyms import PushTGym
from physicalai.policies import Rldx1
from physicalai.train import IterationTimer, Trainer

# Forked DataLoader workers can deadlock/crash under a notebook kernel; spawn is safer.
multiprocessing.set_start_method("spawn", force=True)

MAX_EPOCHS = 5
NUM_WORKERS = 4  # drop to 0-2 if you see worker/decode failures

model = Rldx1(
    pretrained_name_or_path="RLWRLD/RLDX-1-PT",
    gradient_checkpointing=True,
    tune_llm=False,
    tune_visual=False,
    tune_projector=True,
    tune_diffusion_model=True,
    clip_outliers=False,
    tune_top_llm_layers=4,
    tune_vlln=False,
    video_length=1,
    video_stride=1,
    n_action_steps=10,
    backbone_use_lora=True,
    action_model_use_lora=True,
    max_state_dim=2,
    max_action_dim=2,
)

datamodule = LeRobotDataModule(
    repo_id="lerobot/pusht",
    train_batch_size=4,
    data_format="physicalai",
    val_gym=PushTGym(),
    num_workers=NUM_WORKERS,
)

# Save the best checkpoint by validation-gym success rate, keep last too.
best_checkpoint = ModelCheckpoint(
    monitor="val/gym/pc_success",
    mode="max",
    save_top_k=2,
    filename="rldx1-pusht-{epoch:03d}-{val/gym/pc_success:.2f}",
    save_last=True,
    verbose=True,
    save_weights_only=True,
)
lr_monitor = LearningRateMonitor(logging_interval="step")

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    precision="bf16-true",
    log_every_n_steps=20,
    check_val_every_n_epoch=1,
    callbacks=[best_checkpoint, lr_monitor, IterationTimer()],
    limit_train_batches=5,
    limit_val_batches=5,
)

trainer.fit(model=model, datamodule=datamodule)

In [ ]:
best_ckpt_path = best_checkpoint.best_model_path
print(f"Best checkpoint: {best_ckpt_path}")
print(f"Best val/gym/pc_success: {best_checkpoint.best_model_score}")

---
## 2. Evaluate the best checkpoint with `PushTBenchmark`

`PushTBenchmark` renders at the gym-default 96x96 resolution, matching what
`lerobot/pusht` frames look like natively (rendering at a different
resolution is visually out-of-distribution for the trained policy).

In [ ]:
import torch

from physicalai.benchmark.gyms import PushTBenchmark

eval_policy = Rldx1.load_from_checkpoint(best_ckpt_path, map_location="cpu")
eval_policy.eval()
eval_policy.to("cuda")

pusht_video_dir = REPO_ROOT / "pusht_eval_videos"

pusht_benchmark = PushTBenchmark(
    num_episodes=5,
    video_dir=pusht_video_dir,
    record_mode="all",
)

with torch.autocast("cuda", dtype=torch.bfloat16):
    pusht_results = pusht_benchmark.evaluate(eval_policy)

print(pusht_results.summary())

In [ ]:
from IPython.display import Video, display

pusht_video_paths = sorted(pusht_video_dir.glob("*.mp4"))
print(f"Found {len(pusht_video_paths)} PushT episode videos in {pusht_video_dir}")

for video_path in pusht_video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=False, width=320))

---
## 3. LIBERO Benchmark

`lerobot/libero_90`: 90 diverse manipulation tasks in simulation (Franka Panda).
We evaluate the pretrained `RLWRLD/RLDX-1-FT-LIBERO` model on a quick subset
(task suite `libero_10`, task 6) for demo purposes. For full evaluation, expand
to all 10 tasks or all 90 tasks.

In [5]:
import torch
from physicalai.policies import Rldx1
from physicalai.benchmark.gyms import LiberoBenchmark

libero_video_dir = REPO_ROOT / "libero_eval_videos"

libero_policy = Rldx1(
    pretrained_name_or_path="RLWRLD/RLDX-1-FT-LIBERO",
)
libero_policy.to("cuda")
libero_policy.eval()

libero_benchmark = LiberoBenchmark(
    task_suite="libero_10",
    task_ids=[6],  # task 6 from libero_10 suite
    num_episodes=5,
    max_steps=500,
    video_dir=libero_video_dir,
    record_mode="all",
)

with torch.autocast("cuda", dtype=torch.bfloat16):
    libero_results = libero_benchmark.evaluate(libero_policy)

print(libero_results.summary())

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 618.25it/s]



[MSAT Configs] n_cog_tokens: 64
[i] Creating VTC-Qwen3-VL architecture only (weights from checkpoint)
Attention implementation: sdpa

[i] cog_emb initialized in VTCQwen3VLBackbone:
  Shape: torch.Size([64, 4096])
  Dtype: torch.float32
  Min: -0.089677
  Max: 0.094873
  Mean: 0.000039
  Std: 0.019989

[i] Select layers (hs indices): [18] of total_layers=36; kept 18 blocks

Initializing MSAT...
[MSAT] RoPE theta: 10000.0
[MSAT] 'positional_embeddings' of MSAT: rope_sa_only, action_model_max_seq_len: 512, enabled: True
[MSAT] Projecting VL dimension from 4096 to 1536
[MSAT] Output projection: sa_hidden_dim=1536 -> output_dim=1024
[MSAT] Total number of MSAT parameters:  1244626768
[MSAT] Tune action model projector: True
[MSAT] Tune action model diffusion model: True
[MSAT] Tune action model vlln: True
[MSAT] Action model LoRA: False
[load_sharded_weights] Sliced checkpoint param backbone.qwen_model.model.language_model.embed_tokens.weight from [153720, 4096] to [151936, 4096].
[load_sh

The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
/home/yuchunli/git/physical-ai-studio/library/.venv/lib/python3.12/site-packages/transformers/models/qwen2/tokenization_qwen2.py:62: DeprecationWarning: Deprecated in 0.9.0: BPE.__init__ will not create from files anymore, try `BPE.from_file` instead
  BPE(


BENCHMARK RESULTS SUMMARY
Tasks evaluated: 1
Total episodes: 5

AGGREGATE METRICS:
  Success Rate: 100.0%
  Avg Reward: 1.0000
  Avg Episode Length: 239.0
  Avg FPS: 23.0

PER-TASK RESULTS:
  libero_10_6: success=100.0%, reward=1.0000, steps=239.0


In [ ]:
libero_video_paths = sorted(libero_video_dir.glob("*.mp4"))
print(f"Found {len(libero_video_paths)} LIBERO episode videos in {libero_video_dir}")

for video_path in libero_video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=False, width=640))